```markdown
# Analyse Épidémique : Impact des Super-Propagateurs et Stratégies de Contrôle

Ce notebook rassemble l'ensemble des analyses pour le projet ABM (Modélisation Agent-Centrée). L'objectif est d'étudier comment la structure des réseaux de contacts influence la propagation d'une épidémie (modèle SIR) et de comparer l'efficacité de plusieurs politiques sanitaires.

```


In [ ]:
# Importations des bibliothèques standards
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx

# Importation de nos modules locaux modulaires (configurés via __init__.py)
from src import generate_network, ABMSimulation
from src.strategies import evaluer_trade_off


```markdown
## Phases 1 & 2 : Génération des Réseaux et Courbes d'Incidence

Nous étudions trois types de réseaux pour une population de 10 000 agents :
1. **Poisson** : Réseau homogène (les individus ont un nombre de contacts très proche de la moyenne).
2. **Binomiale Négative** : Réseau surdispersé (hétérogénéité modérée).
3. **Power-Law** : Réseau hétérogène sans échelle (présence de très gros hubs capables de générer des événements de super-propagation).

```

---

### Cellule 4 (Code)

In [ ]:
# Taille de la population fixe
N = 10000

# 1. Génération des trois structures de réseaux de contacts
print("[Configuration] Génération des réseaux de contacts...")
networks = {
    'Poisson (Homogène)': generate_network('poisson', n=N, mean_degree=10),
    'Binomiale Négative (Surdispersé)': generate_network('negative_binomial', n=N, r=2, p=0.16),
    'Power-Law (Hétérogène)': generate_network('power_law', n=N, gamma=2.2)
}

# Dictionnaires pour stocker l'historique et les infections secondaires de chaque réseau
results = {}
transmissions = {}

# 2. Lancement des simulations de référence (sans intervention)
plt.figure(figsize=(12, 6))

for name, graph in networks.items():
    # Initialisation de la simulation avec les paramètres de base (beta=0.04, jours d'infection=6)
    sim = ABMSimulation(graph, beta=0.04, recovery_time=6, initial_infected=5)
    history, trans = sim.run(max_days=100)

    results[name] = history
    transmissions[name] = trans

    # Tracé de la courbe d'incidence (nombre d'infectés actifs 'I' au cours du temps)
    plt.plot(history['I'], label=f"{name} (Pic max : {max(history['I'])} cas)")

# Configuration graphique
plt.title("Évolution des courbes d'incidence selon la topologie du réseau", fontsize=14)
plt.xlabel("Temps (Jours)", fontsize=12)
plt.ylabel("Nombre de cas actifs (Infectés)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10)
plt.show()


```markdown
## Phase 2 : Analyse Statistique de la Super-Propagation (Loi de Pareto)

Pour caractériser mathématiquement la super-propagation, nous mesurons :
* **L'Indice de Dispersion** $I_D = \frac{\sigma^2}{\mu}$ des infections secondaires causées par les individus infectés. Un indice très supérieur à 1 indique une forte surdispersion (phénomène de super-propagation).
* **La Courbe de Lorenz** qui illustre visuellement la part cumulée des transmissions générée par la population, permettant de vérifier la règle de Pareto (ex: 20% des individus causent 80% des cas).

```


In [ ]:
plt.figure(figsize=(8, 6))

for name, trans_dict in transmissions.items():
    # Extraction des transmissions causées par chaque agent
    all_transmissions = list(trans_dict.values())

    # Calcul des métriques de dispersion sur l'ensemble de la population
    mean_t = np.mean(all_transmissions)
    var_t = np.var(all_transmissions)
    dispersion_index = var_t / mean_t if mean_t > 0 else 0

    # Tri des transmissions pour construire la courbe de Lorenz
    sorted_trans = np.sort(all_transmissions)
    cumsum_trans = np.cumsum(sorted_trans)
    lorenz_curve = cumsum_trans / cumsum_trans[-1] if cumsum_trans[-1] > 0 else cumsum_trans

    # Calcul de la part des infections causée par le top 20% des propagateurs les plus actifs
    top_20_index = int(len(sorted_trans) * 0.8)
    total_infections = sum(sorted_trans)
    top_20_share = (sum(sorted_trans[top_20_index:]) / total_infections * 100) if total_infections > 0 else 0

    print(f"[{name}]")
    print(f"  -> Indice de dispersion (ID) = {dispersion_index:.2f}")
    print(f"  -> Record de transmission par un seul agent = {max(sorted_trans)}")
    print(f"  -> Le top 20% des agents cause {top_20_share:.1f}% des infections secondaires.\n")

    # Tracé de la courbe de Lorenz correspondante
    x_axis = np.linspace(0, 100, len(lorenz_curve))
    plt.plot(x_axis, lorenz_curve * 100, label=f"{name} (Top 20% = {top_20_share:.1f}% des cas)")

# Ajout de la ligne diagonale d'égalité parfaite (scénario homogène théorique)
plt.plot([0, 100], [0, 100], 'k--', label="Égalité parfaite (Mélange homogène)")

# Configuration graphique
plt.title("Courbe de Lorenz des transmissions secondaires (Validation de Pareto)", fontsize=13)
plt.xlabel("Percentile cumulé de la population (%)", fontsize=11)
plt.ylabel("% Cumulé des infections secondaires générées", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=9)
plt.show()


```markdown
## Phase 3 : Évaluation des Stratégies de Contrôle Sanitaire

Nous appliquons nos scénarios d'intervention sur le réseau **Power-Law**, car sa structure hétérogène le rend particulièrement vulnérable aux super-propagateurs.

L'objectif est d'analyser le compromis (*trade-off*) entre l'efficacité médicale (réduction de la taille de l'épidémie) et le coût social (nombre d'individus contraints à l'isolement ou à la quarantaine).

```


In [ ]:
# Sélection du réseau Power-Law pour le benchmark des stratégies
g_cible = networks['Power-Law (Hétérogène)']

# Exécution du benchmark automatisé via notre module d'analyse des stratégies
df_tradeoff = evaluer_trade_off(g_cible, beta=0.04, recovery_time=6, initial_infected=5)

# Affichage du tableau de données synthétique
print("\n--- TABLEAU RÉCAPITULATIF DES INTERVENTIONS ---")
display(df_tradeoff)

# Création d'un graphique à double axe pour visualiser l'arbitrage
fig, ax1 = plt.subplots(figsize=(12, 6))

# Premier axe (Gauche) : Taille finale de l'épidémie (Histogramme rouge)
color_sanitaire = '#e74c3c'
ax1.set_xlabel('Stratégies de contrôle sanitaire testées', fontsize=12, labelpad=15)
ax1.set_ylabel("Bilan sanitaire (Nombre total d'infectés en fin d'épidémie)", color=color_sanitaire, fontsize=12)
bars = ax1.bar(df_tradeoff.index, df_tradeoff['Taille épidémie'], color=color_sanitaire, alpha=0.6, width=0.4)
ax1.tick_params(axis='y', labelcolor=color_sanitaire)

# Deuxième axe (Droite) : Coût social / Nombre de confinés (Ligne bleue)
ax2 = ax1.twinx()
color_social = '#2980b9'
ax2.set_ylabel("Coût social (Nombre total d'agents isolés)", color=color_social, fontsize=12)
ax2.plot(df_tradeoff.index, df_tradeoff['Coût social (Isolés)'], color=color_social, marker='o', linewidth=2.5, markersize=8)
ax2.tick_params(axis='y', labelcolor=color_social)

# Ajustements et affichage
plt.title("Arbitrage : Efficacité Sanitaire vs Coût Social (Réseau Power-Law)", fontsize=14, pad=20)
fig.tight_layout()
plt.show()

```markdown
## Conclusions et Recommandations de Santé Publique

1. **Impact de la structure** : Le réseau hétérogène (*Power-Law*) engendre des dynamiques épidémiques explosives par rapport aux structures de Poisson. Cela s'explique par son indice de dispersion très élevé ($I_D \gg 1$), confirmant que la majorité des infections provient d'un petit groupe de super-propagateurs.
2. **Incohérence du confinement uniforme** : Restreindre uniformément les contacts à l'ensemble de la population paralyse la société à 100% pour un résultat sanitaire sous-optimal.
3. **Supériorité des mesures ciblées** : Isoler préventivement seulement **5% des individus les plus connectés** (les hubs) détruit la connectivité du réseau et étouffe immédiatement l'épidémie. C'est la stratégie présentant le meilleur ratio coût/bénéfice.
4. **Limite du traçage de contacts** : Bien qu'efficace sur le plan médical, le traçage dynamique autour des cas détectés engendre un coût social élevé et difficilement contrôlable en raison des isolements en chaîne dès qu'un hub est touché.

```